# EDA for iteration 2

Note: please use python version 3.10+ for this

### [2.2] Open electricity

In [ ]:
# Library
import os
import pandas as pd
import asyncio
from datetime import datetime, timedelta
from openelectricity import AsyncOEClient
from openelectricity.client import OEClient
from openelectricity.types import DataMetric

# API key
os.environ["OPENELECTRICITY_API_KEY"] = ""

async def fetch_energy_data():
    start_date = datetime.now() - timedelta(days=300)
    end_date = datetime.now()

    async with AsyncOEClient() as client:
        response = await client.client.get(
            "/data/network/NEM",
            params={
                "metrics": ["energy", "power", "emissions", "price", "emissions_intensity"],
                "interval": "1d",
                "date_start": start_date.isoformat(),
                "date_end": end_date.isoformat(),
                "secondary_grouping": "fueltech_group"
            }
        )
        data = await response.json()
        raw_data = []

        for series in data.get("data", []):
            metric = series.get("metric")
            unit = series.get("unit")
            for result in series.get("results", []):
                fueltech = result.get("columns", {}).get("fueltech_group", "unknown")
                for point in result.get("data", []):
                    raw_data.append({
                        "timestamp": point[0],
                        "value": point[1],
                        "fueltech_group": fueltech,
                        "metric": metric,
                        "unit": unit
                    })

        df = pd.DataFrame(raw_data)
        display(df.head())
        return df


df_energy = await fetch_energy_data()

In [ ]:
def classify_energy(source):
    if source in ['wind', 'solar', 'hydro', 'bioenergy']:
        return 'renewable'
    elif source in ['coal', 'gas', 'distillate']:
        return 'fossil'
    elif source in ['battery_charging', 'battery_discharging', 'pumps']:
        return 'storage'
    else:
        return 'other'

df_energy['category'] = df_energy['fueltech_group'].apply(classify_energy)

In [ ]:
df_energy.to_csv("energy_by_fueltech.csv", index=False)

### [2.1] Property level energy consumption

In [ ]:
# File path
property_path = "./dataprep/1/property-level-energy-consumption-modelled-on-building-attributes-baseline-2011-.csv"

df_property = pd.read_csv(property_path)

In [ ]:
cols_keep = ['property_id', 'geo_point_2d', 'floor_area', 'p_res_2026', 'p_com_2026', 'total_2026' ]
df_property = df_property[cols_keep]

# Remove missing values or not enough value
df_property = df_property[(df_property['floor_area'] > 0) & (df_property['total_2026'] > 0)]

# Additional column to understand consumption energy in each block
df_property['energy_per_m2_2026'] = df_property['total_2026'] / df_property['floor_area']

In [ ]:
df_property.to_csv("property_level_energy.csv", index=False)

### [2.3] [2.4] Block energy retrofit/business-as-usual

In [ ]:
retrofit_path = "./dataprep/4/block-level-energy-consumption-modelled-on-building-attributes-2026-projection-r.csv"
business_path = "./dataprep/4/block-level-energy-consumption-modelled-on-building-attributes-2026-projection-b.csv"

In [ ]:
df_retrofit = pd.read_csv(retrofit_path)
df_business = pd.read_csv(business_path)

In [ ]:
# add column to identify the scenario
df_retrofit["scenario"] = "retrofit"
df_business["scenario"] = "business_as_usual"

# Columns to keep
common_cols = ['Geo Point', 'Geo Shape', 'total', 'scenario']
df_r = df_retrofit[common_cols]
df_b = df_business[common_cols]

# combine the two dataframes
df_merged = pd.merge(
    df_b, df_r,
    on="Geo Point",
    suffixes=("_b", "_r")
)

# Calculate the difference between the two scenarios
df_merged["delta_total"] = df_merged["total_b"] - df_merged["total_r"]

In [ ]:
def categorize_saving(delta):
    if delta > 5000:
        return "Very High"
    elif delta > 1000:
        return "High"
    elif delta > 200:
        return "Moderate"
    elif delta > 0:
        return "Low"
    else:
        return "No Gain"

df_merged["impact_level"] = df_merged["delta_total"].apply(categorize_saving)

df_merged[['lat', 'lon']] = df_merged['Geo Point'].str.split(',', expand=True).astype(float)

In [ ]:
# EDA 
impact_counts = df_merged["impact_level"].value_counts().reset_index()
impact_counts.columns = ["impact_level", "count"]
impact_counts

In [ ]:
df_merged.to_csv("block_level.csv", index=False)